### geemap_AKB is some development code to select Landsat images of SEAN glaciers and download.

In [1]:
import ee
import geemap

In [2]:
#this or ee.Initialize
Map = geemap.Map()

In [13]:
glacier = ee.Geometry.Point(-137.121, 58.838) #Johns Hopkins
#TODO: explore polygon instead (may work seamlessly, may not)
#SEE ALSO: reducing/clipping to area in "create an image composite" section of reducing_image_collection.ipynb
Map.centerObject(glacier, 12)  # Zoom level 12 for a close view
# Add a marker at the point (optional, for visualization)
Map.addLayer(glacier, {'color': 'red'}, 'glacier')

collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    .filterDate('2014-01-01', '2015-01-01')
    .filterBounds(glacier)
)
collectionSR = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterDate('2014-01-01', '2015-01-01')
    .filterBounds(glacier)
)
collection.size().getInfo() 

25

In [14]:
collection.aggregate_array("system:id").getInfo()

#CONCLUSION: top of atmosphere has 3 extra images (25 total) compared to surface reflectance (22 total):  
 #'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20141224', #extra
 #'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20141113', #extra
 #'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20141129'] #extra
#100% Cloudy????

['LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140207',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140223',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140412',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140428',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140615',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140701',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140802',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140818',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140903',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140919',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20141021',
 'LANDSAT/LC08/C02/T1_TOA/LC08_059019_20141224',
 'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20140214',
 'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20140302',
 'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20140318',
 'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20140403',
 'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20140419',
 'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20140505',
 'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20140521',
 'LANDSAT/LC08/C02/T1_TOA/LC08_060019_20140606',
 'LANDSAT/LC08/C02/T

In [23]:
#geemap metadata about the image
first_image = collection.first()
first_imageSR = collectionSR.first()
first_info = geemap.image_props(first_image).getInfo() #DATE_ACQUIRED
first_infoSR = geemap.image_props(first_imageSR).getInfo() #DATE_ACQUIRED
first_info

{'CLOUD_COVER': 27.25,
 'CLOUD_COVER_LAND': 37.08,
 'COLLECTION_CATEGORY': 'T1',
 'COLLECTION_NUMBER': 2,
 'DATA_SOURCE_ELEVATION': 'GLS2000',
 'DATA_SOURCE_TIRS_STRAY_LIGHT_CORRECTION': 'TIRS',
 'DATE_ACQUIRED': '2014-02-07',
 'DATE_PRODUCT_GENERATED': 1599881798000,
 'DATUM': 'WGS84',
 'EARTH_SUN_DISTANCE': 0.9863668,
 'ELLIPSOID': 'WGS84',
 'GEOMETRIC_RMSE_MODEL': 9.891,
 'GEOMETRIC_RMSE_MODEL_X': 6.514,
 'GEOMETRIC_RMSE_MODEL_Y': 7.442,
 'GEOMETRIC_RMSE_VERIFY': 8.942,
 'GRID_CELL_SIZE_PANCHROMATIC': 15,
 'GRID_CELL_SIZE_REFLECTIVE': 30,
 'GRID_CELL_SIZE_THERMAL': 30,
 'GROUND_CONTROL_POINTS_MODEL': 280,
 'GROUND_CONTROL_POINTS_VERIFY': 73,
 'GROUND_CONTROL_POINTS_VERSION': 5,
 'IMAGE_DATE': '2014-02-07',
 'IMAGE_QUALITY_OLI': 9,
 'IMAGE_QUALITY_TIRS': 9,
 'K1_CONSTANT_BAND_10': 774.8853,
 'K1_CONSTANT_BAND_11': 480.8883,
 'K2_CONSTANT_BAND_10': 1321.0789,
 'K2_CONSTANT_BAND_11': 1201.1442,
 'LANDSAT_PRODUCT_ID': 'LC08_L1TP_059019_20140207_20200912_02_T1',
 'LANDSAT_SCENE_ID': 'LC8

In [25]:
first_date=first_info['DATE_ACQUIRED']
first_dateSR=first_infoSR['DATE_ACQUIRED']
first_date+' '+first_dateSR

'2014-02-072014-02-07'

In [26]:
#ee.Image.getInfo() to retrieve metadata about the image
image_info = first_image.getInfo()
#NOTE: from grok "Use getInfo() Sparingly: Calling getInfo() forces a synchronous server request, which can be slow and may hit API limits. Use it for debugging or when you need specific metadata in Python.
#Alternative: For large-scale processing, prefer server-side operations (e.g., ee.Image.get() or ee.Image.select()) to avoid fetching data to the client."

image_info
#CONCLUSION: similar lists - from geemap vs from ee.

{'type': 'Image',
 'bands': [{'id': 'B1',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [8201, 8271],
   'crs': 'EPSG:32608',
   'crs_transform': [30, 0, 289785, 0, -30, 6631515]},
  {'id': 'B2',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [8201, 8271],
   'crs': 'EPSG:32608',
   'crs_transform': [30, 0, 289785, 0, -30, 6631515]},
  {'id': 'B3',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [8201, 8271],
   'crs': 'EPSG:32608',
   'crs_transform': [30, 0, 289785, 0, -30, 6631515]},
  {'id': 'B4',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [8201, 8271],
   'crs': 'EPSG:32608',
   'crs_transform': [30, 0, 289785, 0, -30, 6631515]},
  {'id': 'B5',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [8201, 8271],
   'crs': 'EPSG:32608',
   'crs_transform': [30, 0, 289785, 0, -30, 6631515]},
  {'id': 'B6',
   'data_type': {'type': 'Pi

In [27]:
# Print selected metadata
print("Image ID:", image_info['id'], first_info['system:id'])
print("Date Acquired:", image_info['properties']['DATE_ACQUIRED'], first_info['DATE_ACQUIRED'])
print("Cloud Cover:", image_info['properties']['CLOUD_COVER'], first_info['CLOUD_COVER'])
print("Band Names:", [band['id'] for band in image_info['bands']])
print("Band Names2:", first_info['system:band_names'])
print("Band NamesSR:", first_infoSR['system:band_names'])

Image ID: LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140207 LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140207
Date Acquired: 2014-02-07 2014-02-07
Cloud Cover: 27.25 27.25
Band Names: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'QA_PIXEL', 'QA_RADSAT', 'SAA', 'SZA', 'VAA', 'VZA']
Band Names2: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11', 'QA_PIXEL', 'QA_RADSAT', 'SAA', 'SZA', 'VAA', 'VZA']
Band NamesSR: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']


In [36]:
# Add the Landsat image to the map (visualize with true color bands) USE FOR SURFACE REFLECTANCE
vis_paramsSR = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # Red, Green, Blue
    'min': 2000, #was 8000
    'max': 150000 #was 18000
}
# Add the Landsat TOA image to the map (visualize with true color bands) TOP OF ATMOSPHERE
vis_params = {
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue for TOA
    'min': 0.05,
    'max': 1.6 #was 0.3 or 0. for normal land scenes, not snow
}
Map.addLayer(first_image, vis_params, "First image "+first_date)
Map.addLayer(first_imageSR, vis_paramsSR, "First image SR "+first_date)
Map
#CONCLUSION: SR product seems to have the highlights washed out - details seen in TOA not present in SR, even with max limit set quite high. 
#Would be nice to be more systematic about this, but enough for now...
#Is there a way to plot the histogram of values? Or set min/max to 5%/95%?

Map(bottom=156209.0, center=[58.72081631228038, -136.79385784539554], controls=(WidgetControl(options=['positi…

## Time Series

In [38]:
dates=collection.aggregate_array("DATE_ACQUIRED").getInfo()
dates

['2014-02-07',
 '2014-02-23',
 '2014-04-12',
 '2014-04-28',
 '2014-06-15',
 '2014-07-01',
 '2014-08-02',
 '2014-08-18',
 '2014-09-03',
 '2014-09-19',
 '2014-10-21',
 '2014-12-24',
 '2014-02-14',
 '2014-03-02',
 '2014-03-18',
 '2014-04-03',
 '2014-04-19',
 '2014-05-05',
 '2014-05-21',
 '2014-06-06',
 '2014-08-25',
 '2014-10-12',
 '2014-10-28',
 '2014-11-13',
 '2014-11-29']

In [39]:
Map.add_time_slider(collection, vis_params, labels=dates, time_interval=1)
Map

Map(bottom=156950.0, center=[58.455636630802694, -136.21023422323353], controls=(WidgetControl(options=['posit…